# Traveling Salesperson Problem (TSP)

The Traveling Salesperson Problem (TSP) is a classical optimization problem in which a salesperson must visit a set of cities exactly once and return to the starting city while minimizing the total travel distance.

This project uses **Hill Climbing with Random Restart** to search for a short tour by iteratively improving the visiting order of points.

## Distance Calculation and Tour Representation

Each city is represented by a two-dimensional coordinate (x, y). The `dist()` function calculates the Euclidean distance between two points.

A tour is represented as a list of point indices, where each index refers to a city in the original points list. The `tour_cost()` method calculates the total distance of the complete tour, including the return from the last city to the first one.

## Hill Climbing with Random Restart

The algorithm starts with an initial visiting order and attempts to reduce the total tour distance.

In each Hill Climbing step, `transToBestNeighbor()` swaps pairs of cities and evaluates the resulting tour. If a swap reduces the total cost, the best improving swap is applied. The process continues until no better neighboring tour is found.

Since Hill Climbing may become trapped in a local optimum, `findMinimum()` repeats the search from multiple randomized tours and keeps the best result found across the attempts.

## Implementation and Execution

The `Map` class manages the points, tour, cost calculation, neighbor selection, and optimization process. The `makeTourFromZero()` method rotates the final tour so that it starts from the first point, without changing its cyclic order or total cost.

The example below creates a map with four points, performs 100 search attempts, and displays the resulting tour and its total cost.

In [ ]:
import random

def dist(p1, p2):  # calculates distance of two points
    return ((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)**0.5


class Map:
    def __init__(self, points):
        self.points = list(points)
        self.tour = list(range(len(points)))  # initial_path consists of point indexes

    def randomize(self):
        random.shuffle(self.tour)

    def tour_cost(self):  # calculates the cost of current tour
        cost = 0
        for i in range(len(self.tour)):
            cost += dist(
                self.points[self.tour[i]],
                self.points[self.tour[(i + 1) % len(self.tour)]]
            )
        return cost

    """
    For TSP, the Hill Climbing step swaps two points in the tour.
    """
    def transToBestNeighbor(self):  # hill climbing step
        c1_idx = -1
        c2_idx = -1
        best_cost = self.tour_cost()

        for i in range(len(self.tour) - 1):
            for j in range(i + 1, len(self.tour)):
                self.tour[i], self.tour[j] = self.tour[j], self.tour[i]
                current_cost = self.tour_cost()

                if current_cost < best_cost:
                    best_cost = current_cost
                    c1_idx = i
                    c2_idx = j

                self.tour[i], self.tour[j] = self.tour[j], self.tour[i]

        if c1_idx != -1:
            self.tour[c1_idx], self.tour[c2_idx] = self.tour[c2_idx], self.tour[c1_idx]
            return True  # could climb

        return False  # either reached a flat point or a local maximum (could be global)

    def findMinimum(self, n=10):  # number of randomized starting tours
        best_tour = self.tour
        best_cost = self.tour_cost()

        for i in range(n):
            self.randomize()

            while self.transToBestNeighbor():
                pass

            if self.tour_cost() < best_cost:
                best_tour = self.tour
                best_cost = self.tour_cost()

        self.tour = best_tour

    def print_tour(self):  # prints points
        for i in self.tour:
            print(self.points[i], end=' ')
        print('\n')

    def makeTourFromZero(self):  # changes the tour so it starts from point 0
        idx = self.tour.index(0)
        new_tour = []

        for i in range(len(self.tour)):
            new_tour.append(self.tour[(i + idx) % len(self.tour)])

        self.tour = new_tour


# Create a map with four points
problem = Map([(0, 0), (0, 1), (1, 1), (1, 0)])

# Find a minimum-cost tour using 100 randomized attempts
problem.findMinimum(100)

# Rotate the tour to start from point 0
problem.makeTourFromZero()

# Display the resulting tour and its cost
problem.print_tour()
print('cost:', problem.tour_cost())